In [14]:
%pip install -U scikit-learn > None # Restart environment after
%pip install scikit-learn 



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer, make_classification, load_iris
from sklearn.svm import SVC 
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, ElasticNet 
from sklearn.metrics import mean_squared_error, classification_report 
from sklearn.model_selection import train_test_split, cross_val_score 
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler, StandardScaler 
from sklearn.pipeline import Pipeline 
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score

import matplotlib.pyplot as plt 
from matplotlib.colors import ListedColormap 
import seaborn as sns 

import warnings 
warnings.simplefilter('ignore')

In [16]:
RANDOM_STATE = 2026

In [17]:
data = pd.read_csv('traiinns.csv')
data.head(20)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [18]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


Для оценки моделей лучше всего подходит F1-мера, так как она сочетает в себе методы Recall и Precision, а также идеально подходит под дисбаланс классов(так как в данном датасете погибших больше, чем выживших). В качестве дополнительной метрики вывыдем Accuracy, чтобы сравнить долю правильных ответов умной модели с базовым константным бейслайном 

Подготовка данных 

In [19]:
data['Age'] = data['Age'].fillna(data['Age'].median())

In [20]:
data = pd.get_dummies(data, columns=['Sex'], drop_first=True)
data.head()

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Sex_male
0,1,0,3,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.2500,NaN,S,True
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,71.2833,C85,C,False
2,3,1,3,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,False
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1000,C123,S,False
4,5,0,3,"Allen, Mr. William Henry",35.0,0,0,373450,8.0500,NaN,S,True


In [21]:
features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male']
X = data[features]
y = data['Survived']

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state= RANDOM_STATE, stratify=y)

Начало обучения модели 

In [23]:
dummy_clf = DummyClassifier(strategy = 'most_frequent')
dummy_clf.fit(X_train, y_train)
y_pred_dummy = dummy_clf.predict(X_test)

In [24]:
print('Качество константного бейслайна')
print(f"Accuracy: {accuracy_score(y_test, y_pred_dummy):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred_dummy, zero_division=0):.4f}")

Качество константного бейслайна
Accuracy: 0.6145
F1-score: 0.0000


На основе этих результатов мы видим базовую точку отсчета данной модели. 
А именно при "глупой" модели мы имеем 61% процент точности, но при этом F1-score выдал результат 0, что очевидно, так как модель не строит логических цепочек, она не предсказала ни одного человека, поэтому её Recall равна нулю, а значит и F1-score стремится(в нашем случае достиг) нулю

In [25]:
pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=RANDOM_STATE)
)

In [26]:
pipe.fit(X_train, y_train)

y_pred_lr = pipe.predict(X_test)

print('\n Качество умной модели')
print(classification_report(y_test, y_pred_lr))


 Качество умной модели
              precision    recall  f1-score   support

           0       0.83      0.82      0.82       110
           1       0.71      0.72      0.72        69

    accuracy                           0.78       179
   macro avg       0.77      0.77      0.77       179
weighted avg       0.78      0.78      0.78       179



Модель логистической регрессии показала точность на отложенной выборке (Accuracy) 78 процента, что выше, чем у dummy-модели, помимо этого ключевая метрика F1-score для выживших выросла от 0 до 72 процентов. 
Это доказывает адекватность модели и её способность находить скрытые закономерности по сравнению с константным бэйслайном 